# 02 — Fusion model selection (Step 2)
Loads only `base_outputs_v1.npz` + `split_v1.json`. **No branch retraining.**

Compares two fusion strategies on the same frozen outputs:
- **CWF** — class-weighted late fusion, weights = relative OOF recall, **cross-fitted** (weights fit on 4 folds, scored on the 5th)
- **MFM** — meta-MLP (12 -> 32 -> 16 -> 6) on concatenated probabilities, cross-fitted the same way, best epoch kept with `copy.deepcopy`

Selection rule (fixed in advance): higher cross-fitted OOF macro-F1 wins; if the 95% patient-cluster bootstrap CI of the difference includes 0, **CWF** is chosen (interpretable weights). The **test set is touched once**, in the last section.

Run from a fresh kernel, top to bottom.

In [2]:
import os
from pathlib import Path
ROOT = Path(r"D:\skin_project"); OUT_DIR = ROOT/"outputs"; CACHE_DIR = ROOT/"cache"
for d in (OUT_DIR, CACHE_DIR, ROOT/"tmp"): d.mkdir(parents=True, exist_ok=True)
os.environ["TEMP"] = os.environ["TMP"] = str(ROOT/"tmp")

import json, copy
import numpy as np, pandas as pd
from scipy.stats import binomtest
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, recall_score, balanced_accuracy_score, roc_auc_score
from common import META

d = np.load(OUT_DIR/"base_outputs_v1.npz", allow_pickle=True)
CLASSES = [str(c) for c in d["classes"]]; C = len(CLASSES)
oof_T, oof_I, te_T, te_I = d["oof_xgb"], d["oof_vit"], d["test_xgb"], d["test_vit"]
y_tr, y_te, fold = d["y_train"], d["y_test"], d["fold"]
assert (oof_T.sum(1) > .99).all() and (oof_I.sum(1) > .99).all(), "empty OOF rows"

sp = json.load(open(OUT_DIR/"split_v1.json"))
meta = pd.read_csv(META); id2pat = dict(zip(meta.img_id, meta.patient_id.astype(str)))
pat_tr = np.array([id2pat[i] for i in sp["train_img_ids"]]); pat_te = np.array([id2pat[i] for i in sp["test_img_ids"]])
assert len(pat_tr) == len(y_tr) and len(pat_te) == len(y_te)
CANCER = np.array([c in ("BCC", "SCC", "MEL") for c in CLASSES])
print(CLASSES, "| train", len(y_tr), "test", len(y_te))

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK'] | train 1837 test 461


## 2a. Metrics, ECE, patient-cluster bootstrap

In [3]:
def macro_f1(y, p):
    cm = np.bincount(y*C + p, minlength=C*C).reshape(C, C); tp = np.diag(cm).astype(float)
    prec = tp/np.maximum(cm.sum(0), 1); rec = tp/np.maximum(cm.sum(1), 1)
    return float(np.mean(2*prec*rec/np.maximum(prec+rec, 1e-12)))

def ece(y, P, bins=15):
    conf, pred = P.max(1), P.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins+1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean()*abs(acc[m].mean() - conf[m].mean())
    return float(e)

def cancer_sens_spec(y, p):
    yt, yp = CANCER[y], CANCER[p]
    return float((yp & yt).sum()/max(yt.sum(), 1)), float((~yp & ~yt).sum()/max((~yt).sum(), 1))

def report(name, y, P):
    p = P.argmax(1); se, sp_ = cancer_sens_spec(y, p)
    return dict(model=name, acc=(p == y).mean(), macroF1=macro_f1(y, p), balAcc=balanced_accuracy_score(y, p),
                AUROC=roc_auc_score(y, P, multi_class="ovr", average="macro"), ECE=ece(y, P), cancer_sens=se, cancer_spec=sp_)

def boot(y, pat, preds, B=2000, seed=0):
    """patient-cluster bootstrap of macro-F1 for each model in preds (dict name->hard preds); returns dict name->array"""
    rng = np.random.default_rng(seed); u, inv = np.unique(pat, return_inverse=True)
    groups = [np.where(inv == i)[0] for i in range(len(u))]
    out = {k: np.empty(B) for k in preds}
    for b in range(B):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(u), len(u))])
        for k, p in preds.items(): out[k][b] = macro_f1(y[idx], p[idx])
    return out

def ci(a): return tuple(np.percentile(a, [2.5, 97.5]).round(3))

def mcnemar(y, pa, pb):
    a_only = int(((pa == y) & (pb != y)).sum()); b_only = int(((pa != y) & (pb == y)).sum())
    n = a_only + b_only
    return a_only, b_only, (1.0 if n == 0 else binomtest(a_only, n, 0.5).pvalue)

## 2b. CWF — weights from relative recall, cross-fitted

In [4]:
def cwf_weights(PT, PI, y):
    rT = recall_score(y, PT.argmax(1), average=None, labels=range(C), zero_division=0)
    rI = recall_score(y, PI.argmax(1), average=None, labels=range(C), zero_division=0)
    wT = np.where(rT + rI > 0, rT/np.maximum(rT + rI, 1e-12), 0.5)
    return wT, rT, rI

def cwf_apply(PT, PI, wT):
    s = wT*PT + (1 - wT)*PI
    return s/s.sum(1, keepdims=True)

cwf_oof = np.zeros_like(oof_T)
for k in range(5):
    tr, va = fold != k, fold == k
    wT, _, _ = cwf_weights(oof_T[tr], oof_I[tr], y_tr[tr])
    cwf_oof[va] = cwf_apply(oof_T[va], oof_I[va], wT)

wT_full, rT, rI = cwf_weights(oof_T, oof_I, y_tr)       # final weights: all OOF rows (used for test)
print(pd.DataFrame({"recall_xgb": rT, "recall_vit": rI, "w_xgb": wT_full, "w_vit": 1 - wT_full}, index=CLASSES).round(2))
print("CWF cross-fitted OOF macro-F1: %.3f" % macro_f1(y_tr, cwf_oof.argmax(1)))

     recall_xgb  recall_vit  w_xgb  w_vit
ACK        0.78        0.73   0.51   0.49
BCC        0.72        0.65   0.53   0.47
MEL        0.38        0.53   0.41   0.59
NEV        0.73        0.74   0.50   0.50
SCC        0.23        0.28   0.45   0.55
SEK        0.40        0.67   0.37   0.63
CWF cross-fitted OOF macro-F1: 0.634


## 2c. Meta-MLP — cross-fitted, best epoch restored with `copy.deepcopy`

In [5]:
import torch, torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

class Meta(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2*C, 32), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3), nn.Linear(16, C))
    def forward(self, x): return self.net(x)

def train_meta(X, y, groups, seed=0, max_epochs=400, patience=40, lr=3e-3):
    torch.manual_seed(seed)
    tr_i, va_i = next(GroupShuffleSplit(1, test_size=0.1, random_state=seed).split(X, y, groups))
    Xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y, dtype=torch.long)
    cw = torch.tensor(compute_class_weight("balanced", classes=np.arange(C), y=y[tr_i]), dtype=torch.float32)
    crit = nn.CrossEntropyLoss(weight=cw); m = Meta(); opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-3)
    best, best_state, bad = -1.0, copy.deepcopy(m.state_dict()), 0
    for ep in range(max_epochs):
        m.train(); opt.zero_grad(); crit(m(Xt[tr_i]), yt[tr_i]).backward(); opt.step()
        m.eval()
        with torch.no_grad(): f1 = macro_f1(y[va_i], m(Xt[va_i]).argmax(1).numpy())
        if f1 > best: best, best_state, bad = f1, copy.deepcopy(m.state_dict()), 0   # snapshot, not a reference
        else:
            bad += 1
            if bad >= patience: break
    m.load_state_dict(best_state); return m.eval()

def meta_predict(models, X):
    with torch.no_grad():
        return np.mean([torch.softmax(m(torch.tensor(X, dtype=torch.float32)), 1).numpy() for m in models], 0)

def fit_meta_ensemble(X, y, groups, seeds=(0, 1, 2)): return [train_meta(X, y, groups, seed=s) for s in seeds]

X_oof = np.hstack([oof_T, oof_I]); X_te = np.hstack([te_T, te_I])
mfm_oof = np.zeros_like(oof_T)
for k in range(5):
    tr, va = fold != k, fold == k
    mfm_oof[va] = meta_predict(fit_meta_ensemble(X_oof[tr], y_tr[tr], pat_tr[tr]), X_oof[va])
    print(f"fold {k}: MFM macro-F1 {macro_f1(y_tr[va], mfm_oof[va].argmax(1)):.3f}")
print("MFM cross-fitted OOF macro-F1: %.3f" % macro_f1(y_tr, mfm_oof.argmax(1)))

fold 0: MFM macro-F1 0.632
fold 1: MFM macro-F1 0.664
fold 2: MFM macro-F1 0.644
fold 3: MFM macro-F1 0.654
fold 4: MFM macro-F1 0.614
MFM cross-fitted OOF macro-F1: 0.646


## 2d. Compare on OOF and apply the pre-registered selection rule

In [6]:
P = {"XGB": oof_T, "ViT": oof_I, "CWF": cwf_oof, "MFM": mfm_oof}
hard = {k: v.argmax(1) for k, v in P.items()}
bs = boot(y_tr, pat_tr, hard, B=2000)
rows = []
for k, v in P.items():
    r = report(k, y_tr, v); r["macroF1_95CI"] = ci(bs[k]); rows.append(r)
print(pd.DataFrame(rows).set_index("model").round(3).to_string())

print("\nper-class recall (OOF):")
print(pd.DataFrame({k: recall_score(y_tr, h, average=None, labels=range(C), zero_division=0) for k, h in hard.items()}, index=CLASSES).round(2))

diff = bs["MFM"] - bs["CWF"]; lo, hi = np.percentile(diff, [2.5, 97.5])
a, b, pval = mcnemar(y_tr, hard["MFM"], hard["CWF"])
print(f"\nMFM - CWF macro-F1 diff: {macro_f1(y_tr, hard['MFM']) - macro_f1(y_tr, hard['CWF']):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
print(f"McNemar (MFM-only correct {a}, CWF-only correct {b}): p = {pval:.3f}")

mfm_better = macro_f1(y_tr, hard["MFM"]) > macro_f1(y_tr, hard["CWF"])
SELECTED = "MFM" if (mfm_better and lo > 0) else "CWF"
print("\nSELECTED FUSION MODEL:", SELECTED, "(CI includes 0 -> CWF by rule)" if not (lo > 0 or hi < 0) else "")

         acc  macroF1  balAcc  AUROC    ECE  cancer_sens  cancer_spec    macroF1_95CI
model                                                                                
XGB    0.663    0.549   0.540  0.866  0.174        0.785        0.813  (0.509, 0.588)
ViT    0.657    0.586   0.601  0.892  0.036        0.786        0.825   (0.55, 0.617)
CWF    0.741    0.634   0.626  0.920  0.094        0.827        0.871  (0.598, 0.668)
MFM    0.724    0.646   0.670  0.910  0.246        0.864        0.873  (0.613, 0.676)

per-class recall (OOF):
      XGB   ViT   CWF   MFM
ACK  0.78  0.73  0.84  0.82
BCC  0.72  0.65  0.78  0.71
MEL  0.38  0.53  0.44  0.67
NEV  0.73  0.74  0.81  0.81
SCC  0.23  0.28  0.24  0.30
SEK  0.40  0.67  0.63  0.72

MFM - CWF macro-F1 diff: +0.012  95% CI [-0.015, +0.040]
McNemar (MFM-only correct 78, CWF-only correct 109): p = 0.028

SELECTED FUSION MODEL: CWF (CI includes 0 -> CWF by rule)


## 2e. Fit final fusion on ALL OOF rows, then evaluate on the test set **once**

In [7]:
cwf_te = cwf_apply(te_T, te_I, wT_full)
meta_models = fit_meta_ensemble(X_oof, y_tr, pat_tr)
mfm_te = meta_predict(meta_models, X_te)

PT = {"XGB": te_T, "ViT": te_I, "CWF": cwf_te, "MFM": mfm_te}
hard_te = {k: v.argmax(1) for k, v in PT.items()}
bs_te = boot(y_te, pat_te, hard_te, B=2000, seed=1)
rows = []
for k, v in PT.items():
    r = report(k, y_te, v); r["macroF1_95CI"] = ci(bs_te[k]); rows.append(r)
print(pd.DataFrame(rows).set_index("model").round(3).to_string())
print("\nper-class recall (test, n per class = %s):" % dict(zip(CLASSES, np.bincount(y_te, minlength=C))))
print(pd.DataFrame({k: recall_score(y_te, h, average=None, labels=range(C), zero_division=0) for k, h in hard_te.items()}, index=CLASSES).round(2))
for other in ("CWF", "MFM"):
    a, b, pv = mcnemar(y_te, hard_te[other], hard_te["ViT"])
    print(f"{other} vs ViT-alone McNemar: {other}-only correct {a}, ViT-only correct {b}, p = {pv:.3f}")
print("selected for all later explanations:", SELECTED)

         acc  macroF1  balAcc  AUROC    ECE  cancer_sens  cancer_spec    macroF1_95CI
model                                                                                
XGB    0.659    0.523   0.534  0.868  0.181        0.817        0.814    (0.455, 0.6)
ViT    0.646    0.649   0.676  0.905  0.053        0.757        0.844  (0.569, 0.707)
CWF    0.746    0.707   0.739  0.925  0.113        0.839        0.887  (0.639, 0.758)
MFM    0.729    0.678   0.731  0.910  0.384        0.843        0.896  (0.608, 0.728)

per-class recall (test, n per class = {'ACK': np.int64(127), 'BCC': np.int64(174), 'MEL': np.int64(7), 'NEV': np.int64(49), 'SCC': np.int64(49), 'SEK': np.int64(55)}):
      XGB   ViT   CWF   MFM
ACK  0.78  0.72  0.83  0.81
BCC  0.76  0.60  0.80  0.76
MEL  0.29  0.86  1.00  1.00
NEV  0.80  0.76  0.88  0.88
SCC  0.16  0.35  0.18  0.12
SEK  0.42  0.78  0.75  0.82
CWF vs ViT-alone McNemar: CWF-only correct 84, ViT-only correct 38, p = 0.000
MFM vs ViT-alone McNemar: MFM-only correc

## 2f. Freeze the choice

In [8]:
sel_oof = {"CWF": cwf_oof, "MFM": mfm_oof}[SELECTED]; sel_te = {"CWF": cwf_te, "MFM": mfm_te}[SELECTED]
np.savez(OUT_DIR/"fusion_v1.npz", selected=SELECTED, classes=np.array(CLASSES), w_xgb=wT_full,
         cwf_oof=cwf_oof, mfm_oof=mfm_oof, cwf_test=cwf_te, mfm_test=mfm_te, sel_oof=sel_oof, sel_test=sel_te)
torch.save([m.state_dict() for m in meta_models], OUT_DIR/"meta_mlp_v1.pt")
json.dump({"selected": SELECTED, "w_xgb": wT_full.tolist(), "classes": CLASSES}, open(OUT_DIR/"fusion_v1.json", "w"), indent=1)
print("saved fusion_v1.npz / meta_mlp_v1.pt / fusion_v1.json")

saved fusion_v1.npz / meta_mlp_v1.pt / fusion_v1.json


Trying to check the sc classes where theyre being identified


In [9]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_tr, cwf_oof.argmax(1), labels=range(C))
print(pd.DataFrame(cm, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]))

          pred ACK  pred BCC  pred MEL  pred NEV  pred SCC  pred SEK
true ACK       509        58         4         3        17        12
true BCC        64       526         4        20        45        12
true MEL         5         4        20         9         0         7
true NEV         6        10         4       158         2        15
true SCC        27        73         3         1        35         4
true SEK        25        21         4        11         6       113


In [10]:
def fit_scale(P, y, grid=np.exp(np.linspace(-1, 2, 31)), iters=3):
    s = np.ones(C)
    for _ in range(iters):
        for c in range(C):
            best_f, best_g = -1, 1.0
            for g in grid:
                s2 = s.copy(); s2[c] = g
                f = macro_f1(y, (P*s2).argmax(1))
                if f > best_f + 1e-9: best_f, best_g = f, g
            s[c] = best_g
    return s

adj_oof = np.zeros_like(cwf_oof)
for k in range(5):
    tr, va = fold != k, fold == k
    s = fit_scale(cwf_oof[tr], y_tr[tr])
    adj_oof[va] = cwf_oof[va]*s
    print(f"fold {k} scales:", dict(zip(CLASSES, s.round(2))))

h = {"CWF": cwf_oof.argmax(1), "CWF+scale": adj_oof.argmax(1)}
bsa = boot(y_tr, pat_tr, h, B=2000)
for k, v in h.items(): print(k, "macro-F1 %.3f" % macro_f1(y_tr, v), ci(bsa[k]))
print(pd.DataFrame({k: recall_score(y_tr, v, average=None, labels=range(C), zero_division=0) for k, v in h.items()}, index=CLASSES).round(2))
dd = bsa["CWF+scale"] - bsa["CWF"]; print("diff CI:", np.percentile(dd, [2.5, 97.5]).round(3))

fold 0 scales: {'ACK': np.float64(0.9), 'BCC': np.float64(0.82), 'MEL': np.float64(1.65), 'NEV': np.float64(1.0), 'SCC': np.float64(1.0), 'SEK': np.float64(1.11)}
fold 1 scales: {'ACK': np.float64(0.82), 'BCC': np.float64(0.74), 'MEL': np.float64(1.65), 'NEV': np.float64(1.0), 'SCC': np.float64(1.35), 'SEK': np.float64(1.0)}
fold 2 scales: {'ACK': np.float64(0.82), 'BCC': np.float64(0.82), 'MEL': np.float64(1.35), 'NEV': np.float64(0.9), 'SCC': np.float64(1.35), 'SEK': np.float64(1.11)}
fold 3 scales: {'ACK': np.float64(0.82), 'BCC': np.float64(1.11), 'MEL': np.float64(1.65), 'NEV': np.float64(1.0), 'SCC': np.float64(2.01), 'SEK': np.float64(1.0)}
fold 4 scales: {'ACK': np.float64(0.67), 'BCC': np.float64(0.82), 'MEL': np.float64(1.35), 'NEV': np.float64(0.82), 'SCC': np.float64(1.65), 'SEK': np.float64(0.9)}
CWF macro-F1 0.634 (np.float64(0.598), np.float64(0.668))
CWF+scale macro-F1 0.639 (np.float64(0.605), np.float64(0.671))
      CWF  CWF+scale
ACK  0.84       0.79
BCC  0.78      

In [11]:
def binary_stats(y, p):
    yt, yp = CANCER[y], CANCER[p]
    tp = (yt & yp).sum(); fn = (yt & ~yp).sum(); tn = (~yt & ~yp).sum(); fp = (~yt & yp).sum()
    return dict(TP=int(tp), FN=int(fn), TN=int(tn), FP=int(fp),
                sens=tp/max(tp+fn, 1), spec=tn/max(tn+fp, 1), ppv=tp/max(tp+fp, 1), npv=tn/max(tn+fn, 1))

hp = {"CWF": cwf_oof.argmax(1), "CWF+scale": adj_oof.argmax(1)}
print(pd.DataFrame({k: binary_stats(y_tr, v) for k, v in hp.items()}).T.round(3).to_string())

# patient-cluster bootstrap of sensitivity / specificity (paired: same resamples for both models)
rng = np.random.default_rng(0); u, inv = np.unique(pat_tr, return_inverse=True)
groups = [np.where(inv == i)[0] for i in range(len(u))]
B = 2000; S = {k: np.empty(B) for k in hp}; P_ = {k: np.empty(B) for k in hp}
for b in range(B):
    idx = np.concatenate([groups[i] for i in rng.integers(0, len(u), len(u))])
    for k, v in hp.items():
        st = binary_stats(y_tr[idx], v[idx]); S[k][b], P_[k][b] = st["sens"], st["spec"]

for name, arr in (("sensitivity", S), ("specificity", P_)):
    for k in hp: print(f"{name:12s} {k:10s} 95% CI", np.percentile(arr[k], [2.5, 97.5]).round(3))
    dd = arr["CWF+scale"] - arr["CWF"]
    print(f"{name:12s} diff (scale - CWF) 95% CI", np.percentile(dd, [2.5, 97.5]).round(3), "\n")

# the same split for the three cancer classes: who gets missed?
for k, v in hp.items():
    miss = ~CANCER[v] & CANCER[y_tr]
    print(k, "missed cancers by true class:", {CLASSES[c]: int((miss & (y_tr == c)).sum()) for c in np.where(CANCER)[0]})

              TP     FN     TN     FP   sens   spec    ppv    npv
CWF        710.0  149.0  852.0  126.0  0.827  0.871  0.849  0.851
CWF+scale  726.0  133.0  823.0  155.0  0.845  0.842  0.824  0.861
sensitivity  CWF        95% CI [0.795 0.858]
sensitivity  CWF+scale  95% CI [0.815 0.874]
sensitivity  diff (scale - CWF) 95% CI [0.004 0.034] 

specificity  CWF        95% CI [0.846 0.896]
specificity  CWF+scale  95% CI [0.813 0.869]
specificity  diff (scale - CWF) 95% CI [-0.044 -0.016] 

CWF missed cancers by true class: {'BCC': 96, 'MEL': 21, 'SCC': 32}
CWF+scale missed cancers by true class: {'BCC': 92, 'MEL': 14, 'SCC': 27}
